# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup
# This training code needs Python 3.11 (piper-phonemize has no wheels for newer versions), and Linux.
# So everything heavy runs in a separate Python 3.11 environment at ./venv, whatever Python runs this notebook.
# Requires `uv` on PATH (https://docs.astral.sh/uv/).
import os, subprocess
# Jupyter points matplotlib at a notebook plugin the 3.11 environment does not have
os.environ["MPLBACKEND"] = "Agg"
import shutil
assert shutil.which("uv"), (
    "uv is not on PATH for this notebook. In WSL run: curl -LsSf https://astral.sh/uv/install.sh | sh ; "
    "source ~/.local/bin/env ; then restart `jupyter lab` from that same terminal."
)
!uv venv --allow-existing --python 3.11 venv
VENV = os.path.abspath("venv/bin/python")
assert os.path.exists(VENV), "Creating the Python 3.11 environment failed. Read the uv output above this line."

os.environ["PYTHONUNBUFFERED"] = "1"

def run(cmd):
    """Run a command, show its output in this cell as it happens, and stop the notebook if it fails."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Step failed (exit code {proc.returncode}). Read the output above this message.")

def run_in_venv(code):
    """Run a snippet of Python inside the 3.11 environment."""
    with open("_step.py", "w") as f:
        f.write(code)
    run([VENV, "_step.py"])

def train(flag):
    """One stage of openWakeWord training, run in the 3.11 environment."""
    run([VENV, "openwakeword/openwakeword/train.py", "--training_config", "my_model.yaml", flag])

# piper-sample-generator v2.0.0: the version openWakeWord's train.py was written against
# (newer releases removed generate_samples.py)
!git clone --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

!git clone https://github.com/dscripka/openwakeword
# torch 2.6+ loads weights-only by default, which rejects this full-model checkpoint
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py

# training dependencies (torch 2.7.1 for CUDA 12.8: RTX 50-series GPUs need it)
!uv pip install -q --python {VENV} torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!uv pip install -q --python {VENV} "numpy<2" "setuptools<81" piper-phonemize==1.1.0 webrtcvad mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 "pyarrow<21" datasets==2.14.6 deep-phonemizer==0.0.19 onnx onnxruntime ai-edge-litert pyyaml "scipy<1.15" tqdm scikit-learn requests soundfile librosa
# openWakeWord itself, without speexdsp-ns (not needed for training and has no wheels)
!uv pip install -q --python {VENV} --no-deps -e ./openwakeword

# Download required models
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# sanity check: these imports are what train.py needs
run_in_venv("import pkg_resources, torch, speechbrain, openwakeword, audiomentations, piper_phonemize; print('ok, cuda:', torch.cuda.is_available())")

In [ ]:
# Imports (only what the notebook itself needs; the heavy work runs in the 3.11 environment)
import os
import sys
import yaml
from pathlib import Path

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html
run_in_venv("""
import os, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset):
    name = row["audio"]["path"].split("/")[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row["audio"]["array"]*32767).astype(np.int16))
""")

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# One part of the AudioSet .tar files; for full-scale training use more data.
os.makedirs("audioset", exist_ok=True)
fname = "bal_train09.tar"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -q -O audioset/{fname} {link}
!cd audioset && tar -xf {fname}

run_in_venv("""
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
from tqdm import tqdm

# Convert audioset files to 16khz
output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row["audio"]["array"]*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma), 30 second clips
output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1  # increase for a stronger model
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row["audio"]["array"]*32767).astype(np.int16))
""")

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey nudge"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 10000
config["n_samples_val"] = 2000
config["steps"] = 30000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

train("--generate_clips")

In [ ]:
# Step 2: Augment the generated clips

train("--augment_clips")

In [ ]:
# Step 3: Train model

# train.py saves the .onnx, then tries a tflite conversion that needs TensorFlow (skipped on purpose).
# That last step fails with ModuleNotFoundError: onnx_tf, which is expected. Only the .onnx matters.
try:
    train("--train_model")
except RuntimeError:
    if not os.path.exists("my_custom_model/hey_nudge.onnx"):
        raise
    print("\nTraining finished: my_custom_model/hey_nudge.onnx was saved (the tflite error above is expected).")

In [ ]:
# Step 4 (not needed): the .onnx model from step 3 is what Nudge uses.
# It is saved as my_custom_model/hey_nudge.onnx. Download it from the file browser on the left.
print(os.listdir("my_custom_model"))

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!